Load Data

In [1]:
import pandas as pd
import numpy as np
train_path = r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Data\campaign_train.csv"
test_path = r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Data\campaign_test.csv"
train = pd.read_csv(train_path) 
test = pd.read_csv(test_path) 

Handling missing values

In [2]:
e_level = train["고객_교육수준"].mode()[0]
m_status = train["고객_결혼여부"].mode()[0]
s_mean = train["고객_소득"].median()

e_level, m_status, s_mean

('학사', '기혼', 50827.5)

In [3]:
train["고객_교육수준"] = train["고객_교육수준"].fillna(e_level)
train["고객_결혼여부"] = train["고객_결혼여부"].fillna(m_status)
train["고객_소득"] = train["고객_소득"].fillna(s_mean)

test["고객_교육수준"] = test["고객_교육수준"].fillna(e_level)
test["고객_소득"] = test["고객_소득"].fillna(s_mean)

train.isnull().sum().sum(), test.isnull().sum().sum()

(0, 0)

Feature Engineering 

In [4]:
campaign_cols = ["캠페인1_수락여부", "캠페인2_수락여부", 
                 "캠페인3_수락여부", "캠페인4_수락여부", "캠페인5_수락여부"]

train["과거_캠페인_수락횟수"] = train[campaign_cols].sum(axis=1)
test["과거_캠페인_수락횟수"] = test[campaign_cols].sum(axis=1)

purchase_cols = ['고객_와인_구매금액', '고객_과일_구매금액', '고객_육류_구매금액',
                 '고객_생선_구매금액', '고객_사탕_구매금액', '고객_골드_구매금액']

train["총_구매금액"] = train[purchase_cols].sum(axis=1)
test["총_구매금액"] = test[purchase_cols].sum(axis=1)

train["와인_구매비율"] = train["고객_와인_구매금액"] / (train["총_구매금액"] + 1)
test["와인_구매비율"] = test["고객_와인_구매금액"] / (test["총_구매금액"] + 1)

train["총_웹사이트_방문"] = train["고객_매장방문_구매횟수"] + train["고객_지난달_회사사이트_방문횟수"]
test["총_웹사이트_방문"] = test["고객_매장방문_구매횟수"] + test["고객_지난달_회사사이트_방문횟수"]

KMeans Clustering

In [ ]:
from sklearn.cluster import KMeans

cluster_cols = ["고객_소득", "총_구매금액", "총_웹사이트_방문"]

from sklearn.preprocessing import StandardScaler
scaler_cluster = StandardScaler()

X_cluster_train = scaler_cluster.fit_transform(train[cluster_cols])
X_cluster_test = scaler_cluster.transform(test[cluster_cols])

kmeans = KMeans(n_clusters=4, random_state=42)
train['클러스터'] = kmeans.fit_predict(X_cluster_train)
test['클러스터'] = kmeans.predict(X_cluster_test)

from sklearn.preprocessing import OneHotEncoder

enc_cluster = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_cluster_encoded = pd.DataFrame(enc_cluster.fit_transform(train[['클러스터']]),
                                     columns=enc_cluster.get_feature_names_out(['클러스터']),
                                     index=train.index)
test_cluster_encoded = pd.DataFrame(enc_cluster.transform(test[['클러스터']]),
                                    columns=enc_cluster.get_feature_names_out(['클러스터']),
                                    index=test.index)

train = pd.concat([train, train_cluster_encoded], axis=1)
test = pd.concat([test, test_cluster_encoded], axis=1)

C:\Users\ghwns\AppData\Roaming\Python\Python310\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Excluded the ID and target (Response) columns

In [6]:
drop_cols = ["ID", "target", "고객_가입날짜"]

train_ft = train.drop(columns=drop_cols)
test_ft = test.drop(columns=["ID", "고객_가입날짜"])

One-Hot Encoding

In [ ]:
# Drop unnecessary columns
train_ft = train.drop(columns=["ID", "target", "고객_가입날짜"]).copy()
test_ft = test.drop(columns=["ID", "고객_가입날짜"]).copy()

# One-Hot Encoding for categorical columns
from sklearn.preprocessing import OneHotEncoder
cols = train_ft.select_dtypes("object").columns.tolist()
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

tmp = pd.DataFrame(enc.fit_transform(train_ft[cols]), columns=enc.get_feature_names_out(cols), index=train_ft.index)
train_ft = pd.concat([train_ft.drop(columns=cols), tmp], axis=1)

tmp = pd.DataFrame(enc.transform(test_ft[cols]), columns=enc.get_feature_names_out(cols), index=test_ft.index)
test_ft = pd.concat([test_ft.drop(columns=cols), tmp], axis=1)

# Feature Scaling (Standardization)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Convert to float explicitly before scaling
train_ft = train_ft.astype("float64")
test_ft = test_ft.astype("float64")

train_ft = pd.DataFrame(scaler.fit_transform(train_ft), columns=train_ft.columns, index=train_ft.index)
test_ft = pd.DataFrame(scaler.transform(test_ft), columns=test_ft.columns, index=test_ft.index)

Scaling

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_ft.loc[:, :] = scaler.fit_transform(train_ft)
test_ft.loc[:, :] = scaler.transform(test_ft)

Model

In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

SEED = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
target = train["target"]


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': SEED
    }
    model = LGBMClassifier(**params)
    score = cross_val_score(model, train_ft, target, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    return score


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=30)


best_params = study.best_params
print("Best AUC:", study.best_value)
print("Best Params:", best_params)

clf_lgb = LGBMClassifier(**best_params)
scores = cross_val_score(clf_lgb, train_ft, target, cv=cv, scoring='roc_auc', n_jobs=-1)
print("Final LGBM AUC (Optuna tuned):", np.mean(scores))


clf_lgb.fit(train_ft, target)
pred = clf_lgb.predict_proba(test_ft)[:, 1]

submit = pd.read_csv(r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Submission\campaign_sample_submission.csv")
submit["target"] = pred
submit.to_csv(r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Submission\11.submission.csv", index=False)


[I 2025-06-25 11:21:56,743] A new study created in memory with name: no-name-981f7e38-3105-42c0-8425-91d0e182e57d
[I 2025-06-25 11:22:07,617] Trial 0 finished with value: 0.8570462728874588 and parameters: {'n_estimators': 250, 'learning_rate': 0.09556428757689246, 'num_leaves': 50, 'max_depth': 7, 'min_child_samples': 9, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998}. Best is trial 0 with value: 0.8570462728874588.
[I 2025-06-25 11:22:14,820] Trial 1 finished with value: 0.8539211292423199 and parameters: {'n_estimators': 447, 'learning_rate': 0.0641003510568888, 'num_leaves': 49, 'max_depth': 3, 'min_child_samples': 30, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381}. Best is trial 0 with value: 0.8570462728874588.
[I 2025-06-25 11:22:23,259] Trial 2 finished with value: 0.8598304029724968 and parameters: {'n_estimators': 172, 'learning_rate': 0.026506405886809047, 'num_leaves': 29, 'max_depth': 7, 'min_child_samples': 16, 'subsample

[I 2025-06-25 11:24:21,907] Trial 25 finished with value: 0.8694948479276796 and parameters: {'n_estimators': 318, 'learning_rate': 0.019510393695376674, 'num_leaves': 58, 'max_depth': 5, 'min_child_samples': 10, 'subsample': 0.9994067269571486, 'colsample_bytree': 0.723776288402278}. Best is trial 25 with value: 0.8694948479276796.
[I 2025-06-25 11:24:25,672] Trial 26 finished with value: 0.8616118899869761 and parameters: {'n_estimators': 329, 'learning_rate': 0.018757848561773843, 'num_leaves': 61, 'max_depth': 4, 'min_child_samples': 9, 'subsample': 0.8910046247823353, 'colsample_bytree': 0.7121321738324964}. Best is trial 25 with value: 0.8694948479276796.
[I 2025-06-25 11:24:32,139] Trial 27 finished with value: 0.8588933578487705 and parameters: {'n_estimators': 409, 'learning_rate': 0.027838311671540234, 'num_leaves': 57, 'max_depth': 5, 'min_child_samples': 9, 'subsample': 0.8088742432990841, 'colsample_bytree': 0.7192663010558633}. Best is trial 25 with value: 0.8694948479276

Best AUC: 0.8694948479276796
Best Params: {'n_estimators': 318, 'learning_rate': 0.019510393695376674, 'num_leaves': 58, 'max_depth': 5, 'min_child_samples': 10, 'subsample': 0.9994067269571486, 'colsample_bytree': 0.723776288402278}
Final LGBM AUC (Optuna tuned): 0.860429307438903
[LightGBM] [Info] Number of positive: 200, number of negative: 1144
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003236 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2056
[LightGBM] [Info] Number of data points in the train set: 1344, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.148810 -> initscore=-1.743969
[LightGBM] [Info] Start training from score -1.743969
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f